In [1]:
import os
import sys
import joblib
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

# Adding the project root to the path so I can access our data folder
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
# Loading the TF-IDF matrix we generated in features.py
matrix_path = os.path.join(project_root, "data", "processed", "tfidf_matrix.pkl")
tfidf_matrix = joblib.load(matrix_path)

# Converting from sparse to dense matrix for standard PCA
dense_matrix = tfidf_matrix.toarray()

print(f"Dataset Shape: {dense_matrix.shape}")
print(f"Number of Articles: {dense_matrix.shape[0]}")
print(f"Number of Words (Dimensions): {dense_matrix.shape[1]}")

Dataset Shape: (365, 2000)
Number of Articles: 365
Number of Words (Dimensions): 2000


In [3]:
# Run PCA with 50 components to see how much variance they explain
pca_experiment = PCA(n_components=50)
pca_experiment.fit(dense_matrix)

# Get the variance explained by each individual component
explained_variance = pca_experiment.explained_variance_ratio_

# Calculate the cumulative variance (e.g., Comp 1 + Comp 2 + Comp 3...)
cumulative_variance = explained_variance.cumsum()

# Put it in a DataFrame for easy plotting
variance_df = pd.DataFrame({
    'Number of Components': range(1, 51),
    'Cumulative Explained Variance': cumulative_variance
})

In [4]:
fig = px.line(
    variance_df,
    x='Number of Components',
    y='Cumulative Explained Variance',
    title='PCA Explained Variance (Scree Plot)',
    markers=True,
    template='plotly_dark'
)

# Add a horizontal line at 80% variance (a common benchmark in ML)
fig.add_hline(y=0.80, line_dash="dash", line_color="red", annotation_text="80% Threshold")

fig.show()